In [1]:
print("hello world")

hello world


In [2]:
import os
import json
import joblib
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')

# 1. Initialize Repository Folder Structure
directories = [
    '../data/raw', '../data/processed', '../notebooks', '../src', 
    '../models', '../logs', '../outputs', '../configs', '../reports'
]
for directory in directories:
    os.makedirs(directory, exist_ok=True)
    
print("Project directory structure initialized!")

Project directory structure initialized!


## 1. Load and Inspect Dataset

In [3]:
# Load the raw data
data_path = '../data/raw/churn.csv' 
df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# Check target distribution
print("--- Churn Distribution ---")
print(df["Churn"].value_counts())

class_distribution = df["Churn"].value_counts(normalize=True) * 100
print("\n--- Churn Percentages ---")
print(class_distribution)

--- Churn Distribution ---
Churn
No     5174
Yes    1869
Name: count, dtype: int64

--- Churn Percentages ---
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


## 2. Preprocessing & Data Cleaning
Handling missing values, dropping redundant columns, and encoding the target variable.

In [ ]:
# Handle blank strings in TotalCharges by coercing to NaN and filling with 0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Drop redundant ID column
if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)

# Encode Target Variable (Yes -> 1, No -> 0)
df['Churn'] = df['Churn'].apply(lambda x: 1 if str(x).strip().lower() == "yes" else 0) # pyright: ignore[reportPossiblyUnboundVariable]

# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn'] # type: ignore

print("Data cleaning complete!")

Data cleaning complete!


## 3. Train-Test Split, Feature Engineering & Artifact Export
Splitting the data using stratification, applying scaling/encoding, and immediately saving the processed data and preprocessors to disk to prevent data loss.

In [6]:
# 1. Train-Test Split (Splitting first to avoid data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Separate Column Types
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

# 3. Scale Numerical Features
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train[num_cols])
x_test_scaled = scaler.transform(X_test[num_cols])

# 4. One-Hot Encode Categorical Features
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
x_train_encoded = ohe.fit_transform(X_train[cat_cols])
x_test_encoded = ohe.transform(X_test[cat_cols])

# 5. Combine Features using np.hstack
X_train_final = np.hstack((x_train_scaled, x_train_encoded))
X_test_final = np.hstack((x_test_scaled, x_test_encoded))


# Save Processed Data Arrays 
np.save('../data/processed/X_train_final.npy', X_train_final)
np.save('../data/processed/X_test_final.npy', X_test_final)
np.save('../data/processed/y_train.npy', y_train.values) 
np.save('../data/processed/y_test.npy', y_test.values)

# Save Preprocessors
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(ohe, '../models/ohe.pkl')

# Generate and Save Metadata
metadata = {
    "dataset_name": "Telco Customer Churn",
    "train_shape": X_train_final.shape,
    "test_shape": X_test_final.shape,
    "numerical_features": list(num_cols),
    "categorical_features": list(cat_cols)
}
with open('../data/processed/dataset_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print("Data split, engineered, and immediately saved to disk along with scaler, encoder, and metadata!")

Data split, engineered, and immediately saved to disk along with scaler, encoder, and metadata!


## 4. Baseline Model Development & Evaluation
Training Logistic Regression, Decision Tree, and Random Forest using balanced class weights.

In [7]:
def evaluate_and_store_metrics(model, name, X_test, y_test, results_dict, roc_dict):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] # Probabilities for the positive class (Churn=1)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"--- {name} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}  <-- (How many actual churners did we catch?)")
    print(f"F1-Score : {f1:.4f}\n")
    
    results_dict[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}
    
    # Store ROC data
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    roc_dict[name] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc}

In [8]:
metrics_results = {}
roc_data = {}

# Define Models using class_weight='balanced'
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
}

print("==========================================")
print(" EVALUATION WITH CLASS_WEIGHT='BALANCED'  ")
print("==========================================\n")

# Initialize tracking variables for the tournament
best_f1_score = 0
best_model_name = ""
best_model = None

for model_name, model in models.items():
    # Train and evaluate
    model.fit(X_train_final, y_train)
    evaluate_and_store_metrics(model, model_name, X_test_final, y_test, metrics_results, roc_data)
    
    # Programmatically compare results to find the best model based on F1-Score
    current_f1 = metrics_results[model_name]['F1']
    if current_f1 > best_f1_score:
        best_f1_score = current_f1
        best_model_name = model_name
        best_model = model

# ==========================================
# MODEL EXPORT & ERROR ANALYSIS (TASK 2)
# ==========================================
print("==========================================")
print(f"Best Model: {best_model_name} (F1-Score: {best_f1_score:.4f})")
print("==========================================")

# Save the Champion Model dynamically using its name
file_name = best_model_name.replace(" ", "_").lower()
joblib.dump(best_model, f'../models/{file_name}_baseline.pkl')
print(f"✅ {best_model_name} serialized and saved to models folder!")

# Perform Error Analysis immediately using the dynamic best model
predictions = best_model.predict(X_test_final)
errors_df = X_test.copy()
errors_df['Actual_Churn'] = y_test
errors_df['Predicted_Churn'] = predictions

# Isolate errors
false_negatives = errors_df[(errors_df['Actual_Churn'] == 1) & (errors_df['Predicted_Churn'] == 0)]
false_positives = errors_df[(errors_df['Actual_Churn'] == 0) & (errors_df['Predicted_Churn'] == 1)]

# Save errors for stakeholder review
false_negatives.to_csv('../outputs/false_negatives.csv', index=False)
false_positives.to_csv('../outputs/false_positives.csv', index=False)
print("✅ Error analysis isolated and saved to outputs folder!")

 EVALUATION WITH CLASS_WEIGHT='BALANCED'  

--- Logistic Regression ---
Accuracy : 0.7388
Precision: 0.5052
Recall   : 0.7834  <-- (How many actual churners did we catch?)
F1-Score : 0.6143

--- Decision Tree ---
Accuracy : 0.7346
Precision: 0.5000
Recall   : 0.8075  <-- (How many actual churners did we catch?)
F1-Score : 0.6176

--- Random Forest ---
Accuracy : 0.7580
Precision: 0.5299
Recall   : 0.7807  <-- (How many actual churners did we catch?)
F1-Score : 0.6314

Best Model: Random Forest (F1-Score: 0.6314)
✅ Random Forest serialized and saved to models folder!
✅ Error analysis isolated and saved to outputs folder!
